# Example 1: 使用Modular Strategy执行一个Job

In [1]:
import os

import numpy as np

from ABQflow import BatchAbaqusProcessor, HookSpec, JobSpec, PreparationSpec

%reload_ext autoreload
%autoreload 2

ABAQUS_CAE = 'C:/Applications/SIMULIA/Commands/2026/abaqus.bat'
CWD = os.getcwd()

## 修改inp, 并从该Job的Odb中提取信息

In [2]:
spec1 = JobSpec(
	job_name = "planar_stress_odb",
	workflow = "modular",
	preparation = PreparationSpec(
		kind = "inp_based",
		source_path = "./examples/cae_file/planar_stress_template.inp",
		params = {
			"youngs_modulus": 210000,
			"load_magnitude": 2000,
		}
	),
	post_extraction = [
		HookSpec(
			script_path = "./examples/extraction_scripts/get_max_stress_mises.py",
			tasks = [
				{"result_name": "max_stress_mises",},
				{"result_name": "max_displacement",},
			]
		)
	]
)

import pprint
pprint.pprint(spec1)

JobSpec(job_name='planar_stress_odb',
        workflow='modular',
        preparation=PreparationSpec(kind='inp_based',
                                    source_path='./examples/cae_file/planar_stress_template.inp',
                                    params={'load_magnitude': 2000,
                                            'youngs_modulus': 210000},
                                    options={}),
        preflight=None,
        monolithic_script=None,
        monolithic_params={},
        pre_extraction=[],
        post_extraction=[HookSpec(script_path='./examples/extraction_scripts/get_max_stress_mises.py',
                                  tasks=[{'result_name': 'max_stress_mises'},
                                         {'result_name': 'max_displacement'}],
                                  source='odb')],
        subroutine=None,
        meta={})


In [3]:
processor_odb = BatchAbaqusProcessor(
	batch_data = [spec1],
	base_output_dir = os.path.join(CWD, "examples/01_SingleParameterizedJob/output"),
	cpus_per_job = 4,
	duplicate_mode = "overwrite",
	abaqus_exe = ABAQUS_CAE,
)

In [4]:
outcomes_odb = processor_odb.run_batch(
	num_parallel_jobs = 4,
)

Output()

In [5]:
outcomes_odb

[JobOutcome(job_name='planar_stress_odb', status='COMPLETED', results={'max_stress_mises': 4525.26025390625, 'max_displacement': 3.9895615577697754}, error=None, diagnostics=None, output_dir='c:\\SJTU\\Projects_Code\\24_Abaqus_Pack\\examples/01_SingleParameterizedJob/output\\planar_stress_odb', phases=[{'phase': 'preparation', 'status': 'PREPARATION_SUCCESS', 'started_at': 1785912700.6829748, 'ended_at': 1785912700.6849744, 'duration_s': 0.0019996166229248047, 'error': None}, {'phase': 'simulation', 'status': 'SIMULATION_SUCCESS', 'started_at': 1785912700.6849744, 'ended_at': 1785912757.6265228, 'duration_s': 56.941548347473145, 'error': None}, {'phase': 'post_extraction', 'status': 'EXTRACTION_SUCCESS', 'started_at': 1785912757.6265228, 'ended_at': 1785912760.176402, 'duration_s': 2.549879312515259, 'error': None}], duration_s=59.887511014938354)]

In [6]:
from ABQflow import outcomes_to_dict

for oc in outcomes_odb:
	if oc.status == 'COMPLETED':
		print(oc.job_name, oc.results['max_stress_mises'])
		print(oc.job_name, oc.results['max_displacement'])
	else:
		print(f"{oc.job_name} Fail: {oc.error}")

results_by_name = outcomes_to_dict(outcomes_odb)

planar_stress_odb 4525.26025390625
planar_stress_odb 3.9895615577697754


## 修改inp, 并从inp中提取信息

In [7]:
spec2 = JobSpec(
	job_name = "planar_stress_mass",
	workflow = "modular",
	preparation = PreparationSpec(
		kind = "inp_based",
		source_path = "./examples/cae_file/planar_stress_template.inp",
		params = {
			"youngs_modulus": 210000,
			"load_magnitude": 2000,
		}
	),
	pre_extraction = [
		HookSpec(
			script_path = "./examples/extraction_scripts/get_total_mass.py",
			tasks = [
				{"result_name": "total_mass",},
			]
		)
	]
)

In [8]:
processor_mass = BatchAbaqusProcessor(
	batch_data = [spec2],
	base_output_dir = os.path.join(CWD, "examples/01_SingleParameterizedJob/output"),
	cpus_per_job = 4,
	duplicate_mode = "overwrite",
	abaqus_exe = ABAQUS_CAE,
)

In [9]:
outcomes_mass = processor_mass.run_batch(
	num_parallel_jobs = 4,
)

Output()

In [10]:
outcomes_mass

[JobOutcome(job_name='planar_stress_mass', status='COMPLETED', results={'total_mass': 0.00032066262255247625}, error=None, diagnostics=None, output_dir='c:\\SJTU\\Projects_Code\\24_Abaqus_Pack\\examples/01_SingleParameterizedJob/output\\planar_stress_mass', phases=[{'phase': 'preparation', 'status': 'PREPARATION_SUCCESS', 'started_at': 1785912904.8555725, 'ended_at': 1785912904.8570776, 'duration_s': 0.001505136489868164, 'error': None}, {'phase': 'pre_extraction', 'status': 'EXTRACTION_SUCCESS', 'started_at': 1785912904.8570776, 'ended_at': 1785912908.4154868, 'duration_s': 3.5584092140197754, 'error': None}, {'phase': 'simulation', 'status': 'SIMULATION_SUCCESS', 'started_at': 1785912908.4154868, 'ended_at': 1785912962.743768, 'duration_s': 54.32828116416931, 'error': None}], duration_s=58.25226664543152)]

In [11]:
from ABQflow import outcomes_to_list

for oc in outcomes_mass:
	if oc.status == 'COMPLETED':
		print(oc.job_name, oc.results['total_mass'])
	else:
		print(f"{oc.job_name} Fail: {oc.error}")

results_by_name = outcomes_to_list(outcomes_mass)   # 需要 list 形式时

planar_stress_mass 0.00032066262255247625


## 修改inp, 并从该Job的dat中提取信息

`.odb` 动辄几十上百 GB，只为取几个数就打开一次并不划算。模板 inp 的 Step 里已经加了：

```
*EL PRINT, ELSET=ALL, FREQUENCY=9999, POSITION=INTEGRATION POINTS, SUMMARY=YES
MISES
```

Abaqus 会把这些值以纯文本额外写进 `<job>.dat`。在 `HookSpec` 上设置 **`source="dat"`**，
ABQflow 就改走 `DatExtractionStrategy`：

- 用**宿主 Python** 解析（不是 `abaqus python`）——纯文本，既不需要求解器也**不占 license token**；
- hook 脚本 `get_dat_results.py` 的写法和 Odb 那个 `get_max_stress_mises.py` 一模一样，
  只是把 `odbAccess` 换成了 `datkit`（解析 `.dat` 的库，ABQflow 会一并复制进作业目录）；
- 返回值与 Odb 提取**完全同构**（标量直接返回，场量走 CSV 侧载），`load_field` / `outcomes_to_dict` 照常可用。

> 注意：`*EL PRINT` 对**混合网格每种单元类型各打印一张表**（本例网格是 CPS3 + CPS4R，共两张），
> 所以 hook 里用 `datkit.select_tables(...)` 把两张表一起取出来再合并。

In [ ]:
spec3 = JobSpec(
	job_name = "planar_stress_dat",
	workflow = "modular",
	preparation = PreparationSpec(
		kind = "inp_based",
		source_path = "./examples/cae_file/planar_stress_template.inp",
		params = {
			"youngs_modulus": 210000,
			"load_magnitude": 2000,
		}
	),
	post_extraction = [
		HookSpec(
			script_path = "./examples/extraction_scripts/get_dat_results.py",
			source = "dat",                     # 关键: 从 .dat 读取, 而非默认的 .odb
			tasks = [
				{"result_name": "max_stress_mises",},
				{"result_name": "mises_field", "output": "file"},
			]
		)
	]
)

In [ ]:
processor_dat = BatchAbaqusProcessor(
	batch_data = [spec3],
	base_output_dir = os.path.join(CWD, "examples/01_SingleParameterizedJob/output"),
	cpus_per_job = 4,
	duplicate_mode = "overwrite",
	abaqus_exe = ABAQUS_CAE,
)

In [ ]:
outcomes_dat = processor_dat.run_batch(
	num_parallel_jobs = 4,
)

In [ ]:
outcomes_dat

In [ ]:
for oc in outcomes_dat:
	if oc.status == 'COMPLETED':
		print(oc.job_name, oc.results['max_stress_mises'])
		print(oc.job_name, oc.results['mises_field'])
	else:
		print(f"{oc.job_name} Fail: {oc.error}")

`.dat` 与 `.odb` 两条路径读到的是同一个量，可以互相印证 —— 上面 Odb 章节的
`max_stress_mises` 应当与这里的 `max_mises` 一致：

> 注意：`.dat` 是**打印输出**，数值按 4 位有效数字格式化（如 `4525.` 对 odb 的 `4525.26`），
> 因此两者只在有效位内一致，不会逐位相同。需要全精度时仍应读 `.odb`；
> `.dat` 的价值在于 odb 太大打不开、或只需要工程精度的批量取数。

In [ ]:
# 需要先运行本 notebook 开头的 Odb 章节

from_odb = outcomes_odb[0].results['max_stress_mises']
from_dat = outcomes_dat[0].results['max_stress_mises']
print(f"max mises  odb = {from_odb}")
print(f"max mises  dat = {from_dat}")
print(f"相对偏差       = {abs(from_odb - from_dat) / abs(from_odb):.2%}")

场量结果是一个 CSV 侧载信封, 用 `load_field` 取回 numpy 数组 —— 与 Odb 场量的用法完全一致:

In [ ]:
from ABQflow import load_field

for oc in outcomes_dat:
	if oc.status != 'COMPLETED':
		continue
	print(oc.results['mises_field'])          # 信封: __file__ / format / shape / columns

	arr = load_field(oc, 'mises_field')       # -> numpy (n, 3): ELEMENT, PT, MISES
	print(arr.shape, arr.dtype)

	top5 = arr[np.argsort(arr[:, -1])[::-1][:5]]
	print('MISES 最大的 5 个积分点 (单元号, 积分点号, MISES):')
	print(top5)

### 直接使用 datkit

`get_dat_results.py` 底层用的是 `ABQflow.datkit` —— 一个通用的 Abaqus 打印表解析器。
需要更自由的取数逻辑时, 可以在 notebook 里直接驱动它, 或写进自己的 hook
(ABQflow 会把 `datkit.py` 和 `hookkit.py` 一起复制进作业目录)。

In [ ]:
from ABQflow import datkit

dat_path = os.path.join(CWD, "examples/01_SingleParameterizedJob/output",
						"planar_stress_dat", "planar_stress_dat.dat")

doc = datkit.parse(dat_path)
print(f"分析完成: {doc['completed']}  | 终止方式: {doc['terminator']}")
print(f"增量数  : {doc['increment_count']}  | 表数: {doc['table_count']}")
print()

# *EL PRINT 对每种单元类型各打印一张表
for t in datkit.select_tables(doc, kind='element', increment='last'):
	print(t['description'])
	print(f"  set={t['set']}  columns={t['value_columns']}  rows={len(t['rows'])}")
	print(f"  MAXIMUM = {dict(datkit.summary(t, 'MAXIMUM'))}"
		  f"  @ element {dict(datkit.summary(t, 'MAXIMUM AT'))}")
	print(f"  MINIMUM = {dict(datkit.summary(t, 'MINIMUM'))}"
		  f"  @ element {dict(datkit.summary(t, 'MINIMUM AT'))}")
	print()

# 也可以自己拼表: 跨两张表合并后取前几行
header, rows = datkit.to_rows(
	datkit.select_tables(doc, kind='element', increment='last'),
	columns=['MISES'])
print(header)
for r in rows[:5]:
	print(r)